<a href="https://colab.research.google.com/github/Kumud-Arora/CS4375-Kmeans-Tweet-Analysis/blob/main/kmeans_tweets.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [62]:
import urllib.request
url = "https://raw.githubusercontent.com/Kumud-Arora/CS4375-Kmeans-Tweet-Analysis/refs/heads/main/Health-Tweets/usnewshealth.txt"
urllib.request.urlretrieve(url, "usnewshealth.txt")

('usnewshealth.txt', <http.client.HTTPMessage at 0x7a6f0c5de570>)

In [63]:
import random
import re

# Load tweets

def load_tweets(filename):
    tweets = []
    with open(filename, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split('|')

            if len(parts) >= 3:
                tweet_text = parts[2]
                tweets.append(tweet_text.strip())
    return tweets

# Preprocessing

def preprocess(tweet):
    # removing urls
    tweet = re.sub(r'http\S+', '', tweet)
    # removing mentions
    tweet = re.sub(r'@\w+', '', tweet)
    # removing "#" symbol
    tweet = re.sub(r'#', '', tweet)
    tweet = tweet.lower()
    # removing numbers
    tweet = re.sub(r'\d+', '', tweet)
    # removing punctuation
    tweet = re.sub(r'[^\w\s]', '', tweet)
    return set(tweet.split())

# Jaccard distance

def jaccard_distance(a, b):
  if len(a | b) == 0:
    return 0
  return 1 - len(a & b) / len(a | b)

# Initializing centroids

def initialize_centroids(tweets, k):
  return random.sample(tweets, k)

# Alligning clusters

def assign_clusters(tweets, centroids):
    clusters = [[] for _ in range(len(centroids))]
    for tweet in tweets:
        distances = [jaccard_distance(tweet, c) for c in centroids]
        min_index = distances.index(min(distances))
        clusters[min_index].append(tweet)
    return clusters

# Update centroids

def update_centroids(clusters, centroids, tweets):
  new_centroids = []

  for cluster in clusters:
    if not cluster:
      new_centroids.append(random.choice(tweets))
      continue

    best_tweet = None
    best_distance = float('inf')

    for t1 in cluster:
      total_distance = 0

      for t2 in cluster:
          total_distance += jaccard_distance(t1, t2)

      if total_distance < best_distance:
          best_distance = total_distance
          best_tweet = t1

    new_centroids.append(best_tweet)

  return new_centroids

# Compute SSE

def compute_sse(clusters, centroids):
  sse = 0

  for i in range(len(clusters)):
      for tweet in clusters[i]:
          d = jaccard_distance(tweet, centroids[i])
          sse += d * d

  return sse

# K means algorithm

def kmeans(tweets, K, max_iterations=100):
    centroids = initialize_centroids(tweets, K)

    for _ in range(max_iterations):

        clusters = assign_clusters(tweets, centroids)

        new_centroids = update_centroids(clusters, centroids, tweets)

        if new_centroids == centroids:
            break

        centroids = new_centroids

    return clusters, centroids

if __name__ == "__main__":
  raw = load_tweets("usnewshealth.txt")
  tweets = [preprocess(t) for t in raw if t.strip()]

  K_values = [5, 10, 15, 20, 25]

  results = []

  for K in K_values:
      clusters, centroids = kmeans(tweets, K)
      sse = compute_sse(clusters, centroids)

      sizes = [len(c) for c in clusters]

      results.append((K, sse, sizes))

      print(f"\nK = {K}")
      print(f"SSE = {sse:.4f}")
      for i, size in enumerate(sizes):
          print(f"Cluster {i+1}: {size}")

      print("\nFINAL TABLE:\n")

      for K, sse, sizes in results:
          size_str = ", ".join([f"{i+1}:{sizes[i]}" for i in range(len(sizes))])
          print(f"K={K} | SSE={sse:.2f} | {size_str}")


K = 5
SSE = 1109.2605
Cluster 1: 320
Cluster 2: 76
Cluster 3: 79
Cluster 4: 415
Cluster 5: 510

FINAL TABLE:

K=5 | SSE=1109.26 | 1:320, 2:76, 3:79, 4:415, 5:510

K = 10
SSE = 1061.1456
Cluster 1: 140
Cluster 2: 150
Cluster 3: 73
Cluster 4: 209
Cluster 5: 127
Cluster 6: 71
Cluster 7: 83
Cluster 8: 321
Cluster 9: 127
Cluster 10: 99

FINAL TABLE:

K=5 | SSE=1109.26 | 1:320, 2:76, 3:79, 4:415, 5:510
K=10 | SSE=1061.15 | 1:140, 2:150, 3:73, 4:209, 5:127, 6:71, 7:83, 8:321, 9:127, 10:99

K = 15
SSE = 1036.1276
Cluster 1: 187
Cluster 2: 26
Cluster 3: 43
Cluster 4: 109
Cluster 5: 124
Cluster 6: 20
Cluster 7: 113
Cluster 8: 69
Cluster 9: 141
Cluster 10: 161
Cluster 11: 85
Cluster 12: 83
Cluster 13: 23
Cluster 14: 63
Cluster 15: 153

FINAL TABLE:

K=5 | SSE=1109.26 | 1:320, 2:76, 3:79, 4:415, 5:510
K=10 | SSE=1061.15 | 1:140, 2:150, 3:73, 4:209, 5:127, 6:71, 7:83, 8:321, 9:127, 10:99
K=15 | SSE=1036.13 | 1:187, 2:26, 3:43, 4:109, 5:124, 6:20, 7:113, 8:69, 9:141, 10:161, 11:85, 12:83, 13:23, 14